In [ ]:
import sys
import os
sys.path.insert(0,os.path.abspath('..'))

In [3]:
import redis.asyncio as redis
from config.settings import settings
import json
import logging

In [ ]:
logger=logging.getLogger(__name__)
_redis:redis.Redis|None=None

In [ ]:
async def newRedisClient():
    global _redis
    try:
        _redis=redis.from_url(settings.REDIS_URI,decode_responses=True)
        await _redis.ping()
        logger.info("Connected to Redis")
    except Exception as e:
        logger.error(f"redis error: {e}")
        _redis=None
        raise e

In [6]:
def getRedis()->redis.Redis|None:
    return _redis

In [7]:
async def cacheGet(key:str)->dict|None:
    r=getRedis()
    if r is None:
        return None
    try:
        val=await r.get(key)
        if val is None:
            return None
        return json.loads(val)
    except Exception as e:
        return None

In [ ]:
async def cacheSet(key:str,value:dict,ttl:int=300):
    r=getRedis()
    if r is None:
        return
    try:
        await r.set(key,json.dumps(value,default=str),ex=ttl)
    except Exception:
        pass

In [9]:
async def rateLimitCheck(userId:str,limit:int=30)->bool:
    r=getRedis()
    if r is None:
        return True
    try:
        key=f"rate-{userId}"
        count=await r.incr(key)
        if count==1:
            await r.expire(key,60)
        if count>limit:
            return False
        return True
    except Exception:
        return True

In [10]:
async def closeRedisClient():
    r=getRedis()
    if r is not None:
        await r.close()